# ELECTRA–BiLSTM–MMCRF for Forensic Feature Extraction
Research snapshot from the supplied notebook. Outputs were cleared; data and checkpoint paths were made repository-relative.
**Read [known limitations](../docs/KNOWN_LIMITATIONS.md) before interpreting results. This snapshot is not the corrected word-level implementation.**


In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
import torch

# Cek apakah CUDA tersedia
print("CUDA Available:", torch.cuda.is_available())

# Cek nama device GPU yang digunakan
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
    print("Total Devices:", torch.cuda.device_count())
    print("Current Device Index:", torch.cuda.current_device())
    print("Device Capabilities:", torch.cuda.get_device_capability(0))
else:
    print("CUDA tidak tersedia. Gunakan CPU.")

In [ ]:
import sys

In [ ]:
!{sys.executable} -m pip install transformers
!{sys.executable} -m pip install scikit-learn
!{sys.executable} -m pip install seaborn
!{sys.executable} -m pip install matplotlib

In [ ]:
from __future__ import annotations

import os
import random
from pathlib import Path
from typing import List, Tuple, Dict

import numpy as np
from torch.optim import AdamW
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
# from torchcrf import CRF
from sklearn.metrics import classification_report, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Resolve the repository root when Jupyter starts at root or notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "data").is_dir():
    raise RuntimeError("Start Jupyter from the repository root or notebooks directory")
(PROJECT_ROOT / "models").mkdir(exist_ok=True)
folder_electra = Path("google/electra-base-discriminator")
data_files = {
    # "train": "datasets/anotasi_baru/train.txt",
    # "test":  "datasets/anotasi_baru/test.txt"
    "train": str(PROJECT_ROOT / "data" / "train_clean.txt"),
    "val": str(PROJECT_ROOT / "data" / "val_clean.txt"),
    "test": str(PROJECT_ROOT / "data" / "test_clean.txt"),
}
labels = ["B-FP", "I-FP", "O"]
label2id: Dict[str, int] = {l: i for i, l in enumerate(labels)}
id2label: Dict[int, str] = {i: l for l, i in label2id.items()}
MAX_LEN = 256
BATCH_SIZE = 8
# LR = 3e-5
EPOCHS = 15
SEED = 42
LSTM_HIDDEN = 128
LSTM_LAYERS = 2
# MAX_LEN = 256
# BATCH_SIZE = 16
# LR = 5e-5
# EPOCHS = 15
# SEED = 42
# LSTM_HIDDEN = 256
# LSTM_LAYERS = 2
BEST_MODEL_PATH = str(PROJECT_ROOT / "models" / "electra_best_model_mmcrf_baseline-1.pt")

In [ ]:
def same_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

same_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def read_conll(path: str) -> Tuple[List[List[str]], List[List[str]]]:
    """Load CoNLL‑2003 file → list of sentences (tokens) and labels"""
    sentences: List[List[str]] = []
    tags: List[List[str]] = []
    with open(path, "r", encoding="utf-8") as f:
        tokens, labels_ = [], []
        for line in f:
            line = line.strip()
            if not line:
                if tokens:
                    sentences.append(tokens)
                    tags.append(labels_)
                    tokens, labels_ = [], []
                continue
            splits = line.split()
            tokens.append(splits[0])
            labels_.append(splits[-1])
        if tokens:  # last sentence
            sentences.append(tokens)
            tags.append(labels_)
    return sentences, tags

In [ ]:
import torch
import torch.nn as nn
from typing import List, Optional


class MMCRF(nn.Module):
    def __init__(
        self,
        num_tags: int,
        batch_first: bool = False,
        label2idx: dict = None,
        mask_penalty: float = 1e4,
        forbid_B_to_B: bool = False,
        forbid_B_to_O: bool = False,
    ) -> None:
        if num_tags <= 0:
            raise ValueError(f"invalid number of tags: {num_tags}")
        if label2idx is None:
            raise ValueError("label2idx must be provided")

        super().__init__()

        self.num_tags = num_tags
        self.label2idx_map = label2idx
        self.batch_first = batch_first
        self.mask_penalty = float(mask_penalty)
        self.forbid_B_to_B = forbid_B_to_B
        self.forbid_B_to_O = forbid_B_to_O

        self.start_transitions = nn.Parameter(torch.empty(num_tags))
        self.end_transitions = nn.Parameter(torch.empty(num_tags))
        self.transitions = nn.Parameter(torch.empty(num_tags, num_tags))

        self.reset_parameters()

        trans_mask, start_mask, end_mask = self._build_masks()
        self.register_buffer("trans_mask", trans_mask)
        self.register_buffer("start_mask", start_mask)
        self.register_buffer("end_mask", end_mask)

    def reset_parameters(self) -> None:
        nn.init.uniform_(self.start_transitions, -0.1, 0.1)
        nn.init.uniform_(self.end_transitions, -0.1, 0.1)
        nn.init.uniform_(self.transitions, -0.1, 0.1)

    def _build_masks(self):
        size = self.num_tags
        P = self.mask_penalty
        idx = self.label2idx_map

        if "O" not in idx:
            raise ValueError("label2idx must contain 'O'")

        O_idx = idx["O"]

        trans = torch.zeros(size, size, dtype=torch.float32)
        start = torch.zeros(size, dtype=torch.float32)
        end = torch.zeros(size, dtype=torch.float32)

        # I-X hanya boleh setelah B-X atau I-X
        for next_tag, next_idx in idx.items():
            if next_tag.startswith("I-"):
                ent = next_tag[2:]
                allowed_prev = {f"B-{ent}", f"I-{ent}"}
                for prev_tag, prev_idx in idx.items():
                    if prev_tag not in allowed_prev:
                        trans[prev_idx, next_idx] = -P

        # start tidak boleh I-X
        for tag, i in idx.items():
            if tag.startswith("I-"):
                start[i] = -P

        # Optional: B -> B dilarang
        if self.forbid_B_to_B:
            for prev_tag, prev_idx in idx.items():
                if prev_tag.startswith("B-"):
                    for next_tag, next_idx in idx.items():
                        if next_tag.startswith("B-"):
                            trans[prev_idx, next_idx] = -P

        # Optional: B -> O dilarang, dan B tidak boleh jadi akhir
        if self.forbid_B_to_O:
            for prev_tag, prev_idx in idx.items():
                if prev_tag.startswith("B-"):
                    trans[prev_idx, O_idx] = -P

            for tag, i in idx.items():
                if tag.startswith("B-"):
                    end[i] = -P

        return trans, start, end
        
    def _masked_transitions(self):
        return self.transitions.masked_fill(
            self.trans_mask.lt(0),
            -self.mask_penalty
        )
    # def _masked_transitions(self):
    #     return self.transitions + self.trans_mask

    def _masked_start(self):
        return self.start_transitions + self.start_mask

    def _masked_end(self):
        return self.end_transitions + self.end_mask

    def forward(
        self,
        emissions: torch.Tensor,
        tags: torch.LongTensor,
        mask: Optional[torch.ByteTensor] = None,
        reduction: str = "sum",
    ) -> torch.Tensor:
        self._validate(emissions, tags=tags, mask=mask)

        if reduction not in ("none", "sum", "mean", "token_mean"):
            raise ValueError(f"invalid reduction: {reduction}")

        if mask is None:
            mask = torch.ones_like(tags, dtype=torch.uint8)

        if self.batch_first:
            emissions = emissions.transpose(0, 1)
            tags = tags.transpose(0, 1)
            mask = mask.transpose(0, 1)

        numerator = self._compute_score(emissions, tags, mask)
        denominator = self._compute_normalizer(emissions, mask)
        llh = numerator - denominator

        if reduction == "none":
            return llh
        if reduction == "sum":
            return llh.sum()
        if reduction == "mean":
            return llh.mean()
        return llh.sum() / mask.float().sum().clamp(min=1.0)

    def decode(
        self,
        emissions: torch.Tensor,
        mask: Optional[torch.ByteTensor] = None
    ) -> List[List[int]]:
        self._validate(emissions, mask=mask)

        if mask is None:
            mask = emissions.new_ones(emissions.shape[:2], dtype=torch.uint8)

        if self.batch_first:
            emissions = emissions.transpose(0, 1)
            mask = mask.transpose(0, 1)

        return self._viterbi_decode(emissions, mask)

    def _validate(
        self,
        emissions: torch.Tensor,
        tags: Optional[torch.LongTensor] = None,
        mask: Optional[torch.ByteTensor] = None
    ) -> None:
        if emissions.dim() != 3:
            raise ValueError(f"emissions must have dimension 3, got {emissions.dim()}")

        if emissions.size(2) != self.num_tags:
            raise ValueError(
                f"expected last dimension of emissions is {self.num_tags}, got {emissions.size(2)}"
            )

        if tags is not None and emissions.shape[:2] != tags.shape:
            raise ValueError(
                f"emissions/tags shape mismatch: {tuple(emissions.shape[:2])} vs {tuple(tags.shape)}"
            )

        if mask is not None:
            if emissions.shape[:2] != mask.shape:
                raise ValueError(
                    f"emissions/mask shape mismatch: {tuple(emissions.shape[:2])} vs {tuple(mask.shape)}"
                )
            no_empty_seq = not self.batch_first and mask[0].all()
            no_empty_seq_bf = self.batch_first and mask[:, 0].all()
            if not no_empty_seq and not no_empty_seq_bf:
                raise ValueError("mask of the first timestep must all be on")

    def _compute_score(
        self,
        emissions: torch.Tensor,
        tags: torch.LongTensor,
        mask: torch.ByteTensor
    ) -> torch.Tensor:
        seq_length, batch_size = tags.shape
        mask = mask.float()

        start_t = self._masked_start()
        trans_t = self._masked_transitions()
        end_t = self._masked_end()

        score = start_t[tags[0]]
        score = score + emissions[0, torch.arange(batch_size, device=tags.device), tags[0]]

        for i in range(1, seq_length):
            score = score + trans_t[tags[i - 1], tags[i]] * mask[i]
            score = score + emissions[i, torch.arange(batch_size, device=tags.device), tags[i]] * mask[i]

        seq_ends = mask.long().sum(dim=0) - 1
        last_tags = tags[seq_ends, torch.arange(batch_size, device=tags.device)]
        score = score + end_t[last_tags]

        return score

    def _compute_normalizer(
        self,
        emissions: torch.Tensor,
        mask: torch.ByteTensor
    ) -> torch.Tensor:
        seq_length = emissions.size(0)

        start_t = self._masked_start()
        trans_t = self._masked_transitions()
        end_t = self._masked_end()

        score = start_t + emissions[0]

        for i in range(1, seq_length):
            broadcast_score = score.unsqueeze(2)
            broadcast_emissions = emissions[i].unsqueeze(1)

            next_score = broadcast_score + trans_t + broadcast_emissions
            next_score = torch.logsumexp(next_score, dim=1)

            score = torch.where(mask[i].unsqueeze(1).bool(), next_score, score)

        score = score + end_t
        return torch.logsumexp(score, dim=1)

    def _viterbi_decode(
        self,
        emissions: torch.FloatTensor,
        mask: torch.ByteTensor
    ) -> List[List[int]]:
        seq_length, batch_size = mask.shape

        start_t = self._masked_start()
        trans_t = self._masked_transitions()
        end_t = self._masked_end()

        score = start_t + emissions[0]
        history = []

        for i in range(1, seq_length):
            broadcast_score = score.unsqueeze(2)
            broadcast_emission = emissions[i].unsqueeze(1)

            next_score = broadcast_score + trans_t + broadcast_emission
            next_score, indices = next_score.max(dim=1)

            score = torch.where(mask[i].unsqueeze(1).bool(), next_score, score)
            history.append(indices)

        score = score + end_t
        seq_ends = mask.long().sum(dim=0) - 1

        best_tags_list = []
        for idx in range(batch_size):
            _, best_last_tag = score[idx].max(dim=0)
            best_tags = [best_last_tag.item()]

            for hist in reversed(history[:seq_ends[idx]]):
                best_last_tag = hist[idx][best_tags[-1]]
                best_tags.append(best_last_tag.item())

            best_tags.reverse()
            best_tags_list.append(best_tags)

        return best_tags_list

In [ ]:
class NERDataset(Dataset):
    def __init__(self, sentences: List[List[str]], tags: List[List[str]], tokenizer: AutoTokenizer, max_len: int):
        self.sentences = sentences
        self.tags = tags
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx: int):
        words = self.sentences[idx]
        labels_ = self.tags[idx]
        assert len(words) == len(labels_)

        # Tokenize and create alignment
        tokens, label_ids = ["[CLS]"], [ -100 ]  # -100 ignored in loss
        for word, label in zip(words, labels_):
            word_tokens = self.tokenizer.tokenize(word)
            if not word_tokens:
                word_tokens = ["[UNK]"]
            tokens.extend(word_tokens)
            # first subword keeps label, rest get -100 for loss; needed for evaluation later
            label_ids.append(label2id[label])
            label_ids.extend([-100] * (len(word_tokens) - 1))
        tokens.append("[SEP]")
        label_ids.append(-100)

        # Truncate / pad
        if len(tokens) > self.max_len:
            tokens = tokens[:self.max_len - 1] + ["[SEP]"]
            label_ids = label_ids[:self.max_len]
        attention_mask = [1]*len(tokens)
        token_ids = self.tokenizer.convert_tokens_to_ids(tokens)

        # Padding
        pad_len = self.max_len - len(token_ids)
        token_ids.extend([self.tokenizer.pad_token_id]*pad_len)
        attention_mask.extend([0]*pad_len)
        label_ids.extend([-100]*pad_len)

        return {
            "input_ids": torch.tensor(token_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(label_ids, dtype=torch.long),
            "words": words,  # for prediction examples
            "orig_labels": labels_,
        }

ELECTRA–BiLSTM–MMCRF

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel


class ElectraBiLSTMMMCRF(nn.Module):
    def __init__(
        self,
        electra_model_path: str,
        label2idx: dict,
        lstm_hidden: int = 256,
        lstm_layers: int = 2,
        dropout: float = 0.3,
        mask_penalty: float = 100.0,
        forbid_B_to_B: bool = True,
        forbid_B_to_O: bool = True,
        alpha_mom: float = 1.5,
        easy_o_threshold: float = 0.99
    ):
        super().__init__()

        self.electra = AutoModel.from_pretrained(electra_model_path)
        self.dropout = nn.Dropout(dropout)

        self.label2idx = label2idx
        self.idx2label = {v: k for k, v in label2idx.items()}
        self.num_tags = len(label2idx)

        self.alpha_mom = alpha_mom
        self.easy_o_threshold = easy_o_threshold

        self.bilstm = nn.LSTM(
            input_size=self.electra.config.hidden_size,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0
        )

        self.fc = nn.Linear(lstm_hidden * 2, self.num_tags)

        # Pakai MMCRF lama Anda
        self.crf = MMCRF(
            num_tags=self.num_tags,
            label2idx=label2idx,
            batch_first=True,
            mask_penalty=mask_penalty,
            forbid_B_to_B=forbid_B_to_B,
            forbid_B_to_O=forbid_B_to_O
        )

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.electra(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        sequence_output = self.dropout(outputs.last_hidden_state.float())
        lstm_out, _ = self.bilstm(sequence_output)
        lstm_out = self.dropout(lstm_out)

        emissions = self.fc(lstm_out)

        if labels is not None:
            # Mask token valid:
            # attention_mask = 1, dan labels bukan -100
            mask = attention_mask.bool() & labels.ne(-100)

            safe_labels = labels.clone()
            safe_labels = safe_labels.masked_fill(~mask, self.label2idx["O"])

            # MMCRF lama mensyaratkan timestep pertama harus aktif.
            # Karena token pertama biasanya [CLS] dan labelnya -100,
            # kita paksa menjadi O.
            mask[:, 0] = True
            safe_labels[:, 0] = self.label2idx["O"]

            # ============================================================
            # 1. CRF LOSS
            # MMCRF lama mengembalikan log-likelihood.
            # Untuk training, loss = negative log-likelihood.
            # ============================================================
            crf_log_likelihood = self.crf(
                emissions=emissions,
                tags=safe_labels,
                mask=mask.byte(),
                reduction="mean"
            )

            crf_loss = -crf_log_likelihood

            # ============================================================
            # 2. MoM AUXILIARY TOKEN-LEVEL LOSS
            # Easy majority O token diabaikan.
            # ============================================================
            ce_loss_fn = nn.CrossEntropyLoss(reduction="none")

            flat_emissions = emissions.reshape(-1, self.num_tags)
            flat_labels = safe_labels.reshape(-1)

            token_losses = ce_loss_fn(flat_emissions, flat_labels)

            probs = torch.softmax(emissions, dim=-1)
            confidence_O = probs[:, :, self.label2idx["O"]].reshape(-1)

            flat_mask = mask.reshape(-1).float()
            is_O_class = flat_labels.eq(self.label2idx["O"])

            is_easy_majority = is_O_class & confidence_O.gt(self.easy_o_threshold)

            mom_weights = torch.ones_like(flat_labels, dtype=torch.float)
            mom_weights[is_easy_majority] = 0.0

            active_mom_loss = token_losses * mom_weights * flat_mask

            valid_tokens_count = flat_mask.sum().clamp(min=1.0)
            mom_loss = active_mom_loss.sum() / valid_tokens_count

            # ============================================================
            # 3. TOTAL LOSS
            # ============================================================
            total_loss = crf_loss + (self.alpha_mom * mom_loss)

            return total_loss

        # ================================================================
        # INFERENCE / DECODE
        # ================================================================
        else:
            mask = attention_mask.bool()

            preds = self.crf.decode(
                emissions=emissions,
                mask=mask.byte()
            )

            return preds

In [ ]:
def align_predictions(predictions: List[List[int]], label_ids: torch.Tensor) -> Tuple[List[str], List[str]]:
    """Remove special tokens & subwords (label==-100) to get true/pred label sequences"""
    preds_flat, true_flat = [], []
    for pred_seq, true_seq in zip(predictions, label_ids.tolist()):
        for p, t in zip(pred_seq, true_seq):
            if t == -100:  # skip subword / special tokens
                continue
            preds_flat.append(id2label[p])
            true_flat.append(id2label[t])
    return true_flat, preds_flat


def compute_token_f1(true_labels: List[str], pred_labels: List[str]) -> float:
    return f1_score(true_labels, pred_labels, labels=labels, average="macro")


def spans_from_labels(labels_: List[str]) -> List[Tuple[int, int]]:
    """Return list of (start, end) inclusive spans for entities with label B‑SI/I‑SI"""
    spans = []
    i = 0
    while i < len(labels_):
        if labels_[i] == "B-FP":
            start = i
            i += 1
            while i < len(labels_) and labels_[i] == "I-FP":
                i += 1
            end = i - 1
            spans.append((start, end))
        else:
            i += 1
    return spans


from typing import List


def exact_entity_level_metrics(true_labels: List[str], pred_labels: List[str]):
    true_spans = set(spans_from_labels(true_labels))
    pred_spans = set(spans_from_labels(pred_labels))
    tp = len(true_spans & pred_spans)
    fp = len(pred_spans - true_spans)
    fn = len(true_spans - pred_spans)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return precision, recall, f1, tp, fp, fn, len(true_spans)

def partial_entity_level_metrics(true_labels: List[str], pred_labels: List[str]):
    true_spans = list(spans_from_labels(true_labels))
    pred_spans = list(spans_from_labels(pred_labels))

    tp = 0
    matched_true = set()
    matched_pred = set()

    for p_idx, p_span in enumerate(pred_spans):

        if len(p_span) == 3:
            p_label, p_start, p_end = p_span
        else:
            p_start, p_end = p_span
            p_label = "ENTITY"

        # Bandingkan dengan setiap entitas asli
        for t_idx, t_span in enumerate(true_spans):
            if t_idx in matched_true:
                continue

            if len(t_span) == 3:
                t_label, t_start, t_end = t_span
            else:
                t_start, t_end = t_span
                t_label = "ENTITY"

            if p_label == t_label and max(p_start, t_start) <= min(p_end, t_end):
                tp += 1
                matched_true.add(t_idx)
                matched_pred.add(p_idx)
                break

    fp = len(pred_spans) - len(matched_pred)

    fn = len(true_spans) - len(matched_true)

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    return precision, recall, f1, tp, fp, fn, len(true_spans)

In [ ]:
def train_epoch(model, dataloader, optimizer, scheduler, label2idx):
    model.train()
    total_loss = 0.0

    for batch in tqdm(dataloader, leave=False):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        loss = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


def evaluate(model, dataloader, label2idx, lambda_o=0.2):
    model.eval()
    total_loss = 0.0
    true_labels, pred_labels = [], []

    with torch.no_grad():
        for batch in tqdm(dataloader, leave=False):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            loss = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )

            total_loss += loss.item()

            preds = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            t, p = align_predictions(preds, labels)
            true_labels.extend(t)
            pred_labels.extend(p)

    avg_loss = total_loss / len(dataloader)
    token_f1 = compute_token_f1(true_labels, pred_labels)

    par_ent_precision, par_ent_recall, par_ent_f1, par_tp, par_fp, par_fn, par_total_entities = partial_entity_level_metrics(
        true_labels, pred_labels
    )

    ex_ent_precision, ex_ent_recall, ex_ent_f1, ex_tp, ex_fp, ex_fn, ex_total_entities = exact_entity_level_metrics(
        true_labels, pred_labels
    )

    return {
        "loss": avg_loss,
        "token_f1": token_f1,

        "entity_precision": par_ent_precision,
        "entity_recall": par_ent_recall,
        "entity_f1": par_ent_f1,
        "tp": par_tp,
        "fp": par_fp,
        "fn": par_fn,
        "total_entities": par_total_entities,

        "entity_precision_ex": ex_ent_precision,
        "entity_recall_ex": ex_ent_recall,
        "entity_f1_ex": ex_ent_f1,
        "tp_ex": ex_tp,
        "fp_ex": ex_fp,
        "fn_ex": ex_fn,
        "total_entities_ex": ex_total_entities,

        "true_labels": true_labels,
        "pred_labels": pred_labels,
    }

In [ ]:
import matplotlib.pyplot as plt
import torch
import random
import seaborn as sns

# 1. Aktifkan Class EarlyStopping
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def step(self, score):
        if self.best_score is None or score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

def main():
    print(f"Device: {DEVICE}")
    # --------------- Load data
    tokenizer = AutoTokenizer.from_pretrained(
        str(folder_electra),
        use_fast=True,
        add_prefix_space=True
    )
    train_sents, train_tags = read_conll(data_files["train"])
    val_sents, val_tags     = read_conll(data_files["val"])
    test_sents, test_tags   = read_conll(data_files["test"])

    print("Jumlah train sentence:", len(train_sents))
    print("Jumlah val sentence  :", len(val_sents))
    print("Jumlah test sentence :", len(test_sents))
    train_ds = NERDataset(train_sents, train_tags, tokenizer, MAX_LEN)
    val_ds   = NERDataset(val_sents, val_tags, tokenizer, MAX_LEN)
    test_ds  = NERDataset(test_sents, test_tags, tokenizer, MAX_LEN)

    # def collate_fn_entity_weight(batch):
    #   out = {k: torch.stack([d[k] for d in batch]) for k in ["input_ids","attention_mask","labels"]}
    #   labels_batch = out["labels"]  # (B, S)

    #   id_B = label2id["B-FP"]; id_I = label2id["I-FP"]; id_O = label2id["O"]
    #   token_weights = build_entity_token_weights(labels_batch, id_B, id_I, id_O, lambda_o=0.2)
    #   out["token_weights"] = token_weights
    #   return out

    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda x: {k: torch.stack([d[k] for d in x]) for k in ["input_ids", "attention_mask", "labels"]})
    val_dl  = DataLoader(val_ds,  batch_size=BATCH_SIZE, shuffle=False,  collate_fn=lambda x: {k: torch.stack([d[k] for d in x]) for k in ["input_ids", "attention_mask", "labels"]})
    test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,  collate_fn=lambda x: {k: torch.stack([d[k] for d in x]) for k in ["input_ids", "attention_mask", "labels"]})
    # --------------- Build model
    model = ElectraBiLSTMMMCRF(str(folder_electra),label2id).to(DEVICE)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}")

    electra_params = list(model.electra.parameters())

    head_params = (
        list(model.bilstm.parameters())
        + list(model.fc.parameters())
        + list(model.crf.parameters())
    )

    optimizer = AdamW([
        {'params': electra_params, 'lr': 2e-5},
        {'params': head_params, 'lr': 2e-3}
    ])

    total_steps = len(train_dl) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)

    best_f1 = 0.0

    # 2. Inisialisasi Early Stopper (patience disesuaikan dengan total EPOCHS)
    early_stopper = EarlyStopping(patience=5, min_delta=1e-4) # min_delta 1e-4 cukup sensitif

    train_losses, val_losses, val_f1s = [], [], []

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_epoch(
            model=model,
            dataloader=train_dl,
            optimizer=optimizer,
            scheduler=scheduler,
            label2idx=label2id
        )

        eval_result = evaluate(model, val_dl, label2id)   # <-- pakai validation
        val_loss = eval_result["loss"]
        val_f1 = eval_result["entity_f1"]

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_f1s.append(val_f1)

        print(
            f"Epoch {epoch} — "
            f"train_loss={train_loss:.4f}, "
            f"val_loss={val_loss:.4f}, "
            f"token_f1={eval_result['token_f1']:.4f}, "
            f"entity_f1_partial={eval_result['entity_f1']:.4f}, "
            f"entity_f1_exact={eval_result['entity_f1_ex']:.4f}"
        )

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            print(f"  ** New best model saved to {BEST_MODEL_PATH}")

        # 3. Panggil step() dari early stopper dan masukkan nilai F1 Validasi
        early_stopper.step(val_f1)

        # # 4. Cek apakah kondisi stop terpenuhi
        if early_stopper.early_stop:
            print(f"\n  ## Early stopping triggered at epoch {epoch}! F1-Score tidak membaik selama {early_stopper.patience} epoch berturut-turut.")
            break # Hentikan proses training

    # =========================================================================
    # VISUALISASI 1: Kurva Konvergensi (Gaya Publikasi Jurnal)
    # =========================================================================
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np

    # Pengaturan Global untuk standar Jurnal Internasional
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']
    plt.rcParams['axes.titlesize'] = 14
    plt.rcParams['axes.labelsize'] = 12
    plt.rcParams['xtick.labelsize'] = 11
    plt.rcParams['ytick.labelsize'] = 11
    plt.rcParams['legend.fontsize'] = 11

    # 1A. Plot Training & Validation Loss
    plt.figure(figsize=(8, 5), dpi=300)
    plt.plot(train_losses, marker='o', linestyle='-', color='#004C99', label='Train Loss')
    plt.plot(val_losses, marker='s', linestyle='--', color='#990000', label='Val Loss')
    plt.xlabel('Epoch', fontweight='bold')
    plt.ylabel('Loss Value', fontweight='bold')
    plt.title('ELECTRA-BiLSTM-MMCRF Convergence: Training vs Validation Loss', fontweight='bold')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    # plt.savefig('baseline_loss_curve.png', bbox_inches='tight')
    plt.show()

    # 1B. Plot Validation F1-Score
    plt.figure(figsize=(8, 5), dpi=300)
    plt.plot(val_f1s, marker='D', linestyle='-', color='#006600', linewidth=2)
    plt.xlabel('Epoch', fontweight='bold')
    plt.ylabel('Entity-Level F1-Score', fontweight='bold')
    plt.title('ELECTRA-BiLSTM-MMCRF Validation F1-Score', fontweight='bold')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    # plt.savefig('baseline_f1_curve.png', bbox_inches='tight')
    plt.show()

    # --------------- Final evaluation
    print("\n==================================================")
    print("  FINAL EVALUATION - MODEL TERBAIK")
    print("==================================================")

    # Reload best model
    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
    model.to(DEVICE)

    eval_result = evaluate(model, test_dl, label2id)

    true_labels = eval_result["true_labels"]
    pred_labels = eval_result["pred_labels"]

    print("\nToken-Level Performance (tanpa subword):")
    print(classification_report(true_labels, pred_labels, labels=labels, digits=4))

    print("\nEntity-Level Performance Partial:")
    print(f"Precision: {eval_result['entity_precision']:.4f} | "
          f"Recall: {eval_result['entity_recall']:.4f} | "
          f"F1: {eval_result['entity_f1']:.4f}")
    print(f"TP: {eval_result['tp']} | FP: {eval_result['fp']} | FN: {eval_result['fn']}")
    print(f"Total Entities: {eval_result['total_entities']}")
    print("\n\nEntity-Level Performance Exact:")
    print(f"Precision: {eval_result['entity_precision_ex']:.4f} | "
          f"Recall: {eval_result['entity_recall_ex']:.4f} | "
          f"F1: {eval_result['entity_f1_ex']:.4f}")
    print(f"Total Entities: {eval_result['total_entities']}")
    print(f"TP: {eval_result['tp_ex']} | FP: {eval_result['fp_ex']} | FN: {eval_result['fn_ex']}")
    print(f"Total Entities: {eval_result['total_entities_ex']}")
    # =========================================================================
    # VISUALISASI 2: Heatmap Transisi Baseline (Pembuktian Kelemahan)
    # =========================================================================
    print("\nMenyiapkan Heatmap Transisi Baseline...")

    # 1. Definisikan urutan kelas agar rapi di grafik
    labels_order = labels
    n_labels = len(labels_order)
    l2i_heat = {l: i for i, l in enumerate(labels_order)}

    # 2. Siapkan matriks kosong
    transition_matrix = np.zeros((n_labels, n_labels), dtype=int)

    # 3. Hitung transisi riil dari prediksi model baseline
    for i in range(len(pred_labels) - 1):
        current_tag = pred_labels[i]
        next_tag = pred_labels[i+1]

        if current_tag in l2i_heat and next_tag in l2i_heat:
            transition_matrix[l2i_heat[current_tag], l2i_heat[next_tag]] += 1

    # 4. Plotting Heatmap (Gunakan warna merah/Reds untuk Baseline)
    plt.figure(figsize=(6, 5), dpi=300)
    ax = sns.heatmap(
        transition_matrix,
        annot=True, fmt='d', cmap='Reds', # Sengaja pakai 'Reds' agar kontras dgn MCRF yg 'Blues'
        xticklabels=labels_order, yticklabels=labels_order,
        cbar_kws={'label': 'Frequency of Predicted Transitions'},
        linewidths=.5, linecolor='black'
    )

    plt.xlabel('Predicted Next Token ($t+1$)', fontsize=12, fontweight='bold')
    plt.ylabel('Predicted Current Token ($t$)', fontsize=12, fontweight='bold')
    plt.title('Baseline Empirical Transition Matrix', fontsize=14)

    # Beri kotak biru tebal pada sel ilegal (O -> I-FP) untuk menyoroti KESALAHAN baseline
    import matplotlib.patches as patches
    ax.add_patch(patches.Rectangle((2, 0), 1, 1, fill=False, edgecolor='blue', lw=3))

    plt.tight_layout()
    plt.savefig('baseline_heatmap_transisi.png', bbox_inches='tight')
    plt.show()

    # ------------- Show random prediction examples
    print("\nContoh Prediksi 100 biji (per kata asli):")
    random.seed(SEED)
    idxs = random.sample(range(len(test_sents)), min(100, len(test_sents))) # Tambahkan min() agar tidak error jika data uji < 500
    for idx in idxs:
        words = test_sents[idx]
        gold = test_tags[idx]
        # Predict
        encoded = tokenizer(words, is_split_into_words=True, return_tensors="pt", padding="max_length", truncation=True, max_length=MAX_LEN).to(DEVICE)
        preds = model(encoded["input_ids"], encoded["attention_mask"])[0]  # single sentence

        # Align predictions (remove special tokens & subwords)
        word_ids = encoded.word_ids(batch_index=0)
        aligned = []
        prev_wid = None
        for p, wid in zip(preds, word_ids):
            if wid is None or wid == prev_wid:  # skip [CLS]/[SEP]/pad & sub-token
                prev_wid = wid
                continue
            aligned.append(id2label[p])
            prev_wid = wid

        # Tampilkan beberapa kalimat saja (di-comment jika terlalu panjang di output)
        print(f"\nKalimat: \"{' '.join(words)}\"")
        print(f"Kata-kata: {words}")
        print(f"Asli     : {gold}")
        print(f"Prediksi : {aligned}")
         # Show detected entities
        spans = spans_from_labels(aligned)
        if spans:
            print("Entitas Terdeteksi:")
            for s, e in spans:
                print(f"- {' '.join(words[s:e+1])} (FP)")
        else:
            print("Entitas Terdeteksi: None")

    print(">> Heatmap Baseline berhasil disimpan sebagai 'baseline_heatmap_transisi.png'")
    print(f"\n>> Checkpoint terbaik disimpan di: {BEST_MODEL_PATH}")

if __name__ == "__main__":
    main()